#### **OPTIMALIDAD DE LA SOLUCION - ALGORITMOS GREEDY**

In [51]:
import pandas as pd 
import matplotlib.pyplot as plt
from importlib import reload

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")
factibilidad = pd.read_csv("4r.factibilidad.csv")

Empecemos guardando los productos y tipos de cajas en listas en el estado actual, para cargarlos luego a las soluciones. Calculamos también las cajas asignables a cada producto en la lista de productos.

In [52]:
def guardar_cajas_y_productos(grosor):
    caja_compras_merge = especificaciones_cajas.merge(procurement_cajas,
                                                  on="caja_tipo_id")

    cajas = [
        Caja(
            caja_id = row["caja_tipo_id"],
            dim_interior_ancho = row["caja_interior_ancho"],
            dim_interior_largo = row["caja_interior_largo"],
            dim_interior_alto = row["caja_interior_alto"],
            costo_unitario = row['costo_unitario_base']
        )
        for _, row in caja_compras_merge.iterrows()
    ]
    operaciones_planta_aux = operaciones_planta.drop('codigo_producto', axis=1) 
    
    prod_op_merge = pd.concat([catalogo_productos, operaciones_planta_aux], axis=1)

    productos = [
        Producto(
            codigo_producto = row['codigo_producto'],
            cantidad_paquetes = row['cantidad_paquetes'],
            peso_paquete = row['peso_neto_paquete'],
            demanda_buenos_aires = row['volumen_producto_planta_buenos_aires'],
            demanda_curitiba = row['volumen_producto_planta_curitiba'],
            demanda_santiago = row['volumen_producto_planta_santiago'],
            demanda_monterrey = row['volumen_producto_planta_monterrey'],
            demanda_bakersfield = row['volumen_producto_planta_bakersfield'],
            dim_producto_ancho = row['dim_producto_ancho'], 
            dim_producto_largo = row['dim_producto_largo'],
            dim_producto_alto = row['dim_producto_alto']
        )
        for _, row in prod_op_merge.iterrows()
    ]
    # Crear un diccionario de cajas por producto desde el CSV de factibilidad
    cajas_por_producto = {}
    for _, row in factibilidad.iterrows():
        codigo = row['codigo_producto']
        cajas_str = row['cajas_asignables_id']
        
        if isinstance(cajas_str, str) and cajas_str:
            # Separar por '; ' y limpiar
            cajas_list = [c.strip() for c in cajas_str.split(';') if c.strip()]
            cajas_por_producto[codigo] = cajas_list

    # Recorrer la lista de productos en orden y agregar las cajas
    for producto in productos:
        if producto.codigo_producto in cajas_por_producto:
            for caja_id in cajas_por_producto[producto.codigo_producto]:
                producto.agregar_caja_asignable(caja_id)
                
    #Elegir grosor
    for caja in cajas:
        caja.elegir_grosor(grosor_mm=grosor)
    return cajas, productos

#### **Greedy 0: Costos base sin descuentos**

Inicialmente, plantearemos una solución únicamente comparando los costos unitarios base, sin considerar todavía los descuentos por volúmenes anuales.

Como no hay ninguna restricción sobre la cantidad de cajas que podemos comprar de cada tipo, el problema se reduce en encontrar para cada producto el tipo de caja que más le convenga, es decir, el que ofrezca el menor costo.

Empecemos eligiendo un grosor de 5mm para todos los tipos de cajas, de manera que la restricción de headspace no supone ningún problema, pues las dimensiones internas de las cajas se diferencian hasta un 10% de las originales.

In [53]:
cajas, productos = guardar_cajas_y_productos(grosor=5)

In [54]:
def buscar_caja_por_id(id):
    caja_a_buscar = None
    for caja in cajas:
        if caja.caja_id == id:
            caja_a_buscar = caja
    return caja_a_buscar

In [55]:
solucion_base = Solucion(grosor=5)

for producto in productos:
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        solucion_base.agregar_asignacion(producto, caja, descuentos=False)

solucion_base.exportar_submmit(nombre_csv="1-base_5mm")
solucion_base.resumen_por_asignacion()

,codigo_producto,demanda_total,caja_id,utilizacion_pallet,utilizacion_caja,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,46538be6ce954d7b010ddb9c9fb00f15,0.886967,0.952830,1082629.10,38667,5800050,6882679.10
1,BR0001,1546613,58c5c51d6e39e80f84be2a39cc4677a8,0.846404,1.000000,1082629.10,38667,5800050,6882679.10
2,BR0001,1546613,845990194eeb50cd131954a55fecd655,0.436688,0.968634,927967.80,77333,11599950,12527917.80
3,BR0001,1546613,895cc0310e1ec8880587a025b4de25c5,0.863889,0.979142,1005298.45,38667,5800050,6805348.45
4,BR0001,1546613,d0823ee866ead1526b62ea77fb86dde9,0.899519,0.939141,1005298.45,38667,5800050,6805348.45
...,...,...,...,...,...,...,...,...,...
6041,BR0421,145803,c8c092bbf8da5f80452c66c38d27305f,0.803833,1.000000,94771.95,2604,390600,485371.95
6042,BR0421,145803,d0fa74a812078a5c185018f80142e8de,0.811290,0.989922,87481.80,2604,390600,478081.80
6043,BR0421,145803,d20c085d39ceb93f1f9dbe59c29a4786,0.858796,0.933333,94771.95,2604,390600,485371.95
6044,BR0421,145803,ef0525f0911fd36073c2b10f4d81dfd5,0.841620,0.953191,94771.95,2604,390600,485371.95


#### **Greedy 1: Elección por menor costo**

Greedy (1)

Ordenamos los productos de menor a mayor según la cantidad de tipos de cajas asignables.
Iteramos los productos p desde el menor:
    (HACER EN FACTIBILIDAD) Ver factibilidad de caja por resistencia a comprension + headspace
    Asignar a p el tipo de caja con menor costo considerando el cálculo de descuentos sumando los volúmenes de producto
    Recalculo descuentos

Con los descuentos finales, vuelvo a calcular cada costo de producto

Guardamos nuevamente cajas y productos, reiniciando toda asignación previa:

In [56]:
cajas, productos = guardar_cajas_y_productos(grosor=5)

Ordenamos los productos según la cantidad de cajas asignables de menor a mayor

In [57]:
def ordenar_segun_cantidad_cajas_asignables(productos):
    cantidad_cajas_asignables = pd.read_csv('cantidad_cajas_asignables.csv')
    orden_productos = cantidad_cajas_asignables['codigo_producto']

    # orden_codigos es la lista con el orden deseado de códigos de producto
    orden_dict = {codigo: i for i, codigo in enumerate(orden_productos)}

    productos_ordenados = sorted(productos, key=lambda p: orden_dict[p.codigo_producto])
    return productos_ordenados

In [58]:
productos_ordenados = ordenar_segun_cantidad_cajas_asignables(productos)

In [59]:
solucion_greedy1 = Solucion(grosor=5)

for producto in productos_ordenados:
    costos_totales = {}
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        asignacion = Asignacion(producto, caja)
        caja.asignar_producto(producto)
        costo_packaging = asignacion.costo_packaging_producto_total()
        costo_flete = 150 * asignacion.cant_pallets_requeridas()
        
        costo_total = costo_packaging + costo_flete
        costos_totales[caja_id] = costo_total
        
        caja.revocar_producto(producto)
    
    caja_id_optima = min(costos_totales, key=costos_totales.get)
    caja_optima = buscar_caja_por_id(caja_id_optima)
    
    solucion_greedy1.agregar_asignacion(producto,caja_optima)
    
solucion_greedy1.exportar_submmit(nombre_csv="-greedy1_5mm")
solucion_greedy1.resumen_por_asignacion()

,codigo_producto,demanda_total,caja_id,utilizacion_pallet,utilizacion_caja,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,d0823ee866ead1526b62ea77fb86dde9,0.899519,0.939141,715185.835,38667,5800050,6515235.835
1,BR0002,139211,057b7f8c3772129fb0b8cccf3c827c80,0.986667,0.909091,63341.005,2901,435150,498491.005
2,BR0003,172506,4c289dfcdb1f9fab8de1224c88f288d0,0.961111,0.990620,89703.120,2157,323550,413253.120
3,BR0004,271715,340bacb8c1792bfc8b3b0346513c0585,0.972222,0.933333,123630.325,4853,727950,851580.325
4,BR0005,7586,6ecff7742d4a01e1ada1f078a109ba73,0.900000,0.946492,3451.630,159,23850,27301.630
...,...,...,...,...,...,...,...,...,...
430,BR0417,2761,97f8ba1fc812d49f9c98d8f46da0614a,0.844404,0.947362,1159.620,50,7500,8659.620
431,BR0418,3942,97f8ba1fc812d49f9c98d8f46da0614a,0.844404,0.947362,1655.640,71,10650,12305.640
432,BR0419,43300,30cfce68c5506ad8e83e32665dba84af,0.941919,0.968672,20784.000,602,90300,111084.000
433,BR0420,205852,623e13a6f27840cacf8833829b96bf96,0.859778,0.976077,93662.660,3217,482550,576212.660


#### **Greedy 2: Elección por mayor volumen**

Greedy (2)

Ordenamos los productos de menor a mayor según la cantidad de tipos de cajas asignables.
Iteramos los productos p desde el menor:
    (HACER EN FACTIBILIDAD) Ver factibilidad de caja por resistencia a comprension + headspace
    Asignar a p el tipo de caja con mayor volúmen actualmente

Con los descuentos finales, vuelvo a calcular cada costo de producto

In [60]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
productos_ordenados = ordenar_segun_cantidad_cajas_asignables(productos)

In [61]:
solucion_greedy2 = Solucion(grosor=5)

for producto in productos_ordenados:
    metricas = {}  # caja_id -> (volumen_total, costo_total)
    
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        
        # volumen requerido actual (antes de asignar este producto)
        volumen_total = caja.unidades_total_requeridas()
        
        # costo simulado de asignar este producto a esta caja
        asignacion = Asignacion(producto, caja)
        caja.asignar_producto(producto)
        costo_packaging = asignacion.costo_packaging_producto_total()
        costo_flete = 150 * asignacion.cant_pallets_requeridas()
        costo_total = costo_packaging + costo_flete
        caja.revocar_producto(producto)
        
        metricas[caja_id] = (volumen_total, costo_total)
    
    # criterio: mayor volumen_total primero, y en caso de empate, menor costo_total
    caja_id_optima = max(metricas, key=lambda id: (metricas[id][0], -metricas[id][1]))
    caja_optima = buscar_caja_por_id(caja_id_optima)
    
    solucion_greedy2.agregar_asignacion(producto, caja_optima)

solucion_greedy2.exportar_submmit(nombre_csv="-greedy2_5mm")
solucion_greedy2.resumen_por_asignacion()

,codigo_producto,demanda_total,caja_id,utilizacion_pallet,utilizacion_caja,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,d0823ee866ead1526b62ea77fb86dde9,0.899519,0.939141,715185.835,38667,5800050,6515235.835
1,BR0002,139211,057b7f8c3772129fb0b8cccf3c827c80,0.986667,0.909091,72389.720,2901,435150,507539.720
2,BR0003,172506,f3108b9695ef6420ab92a29c97d1f222,0.988889,0.961138,89703.120,2157,323550,413253.120
3,BR0004,271715,340bacb8c1792bfc8b3b0346513c0585,0.972222,0.933333,123630.325,4853,727950,851580.325
4,BR0005,7586,6ecff7742d4a01e1ada1f078a109ba73,0.900000,0.946492,3451.630,159,23850,27301.630
...,...,...,...,...,...,...,...,...,...
430,BR0417,2761,2bcfb295bee600aa6914bbd06f5891b6,0.875972,0.912211,1256.255,50,7500,8756.255
431,BR0418,3942,2bcfb295bee600aa6914bbd06f5891b6,0.875972,0.912211,1793.610,71,10650,12443.610
432,BR0419,43300,d66062da2ea02af9a197381fef2d9894,0.995000,0.915344,22516.000,602,90300,112816.000
433,BR0420,205852,623e13a6f27840cacf8833829b96bf96,0.859778,0.976077,93662.660,3217,482550,576212.660


#### **Greedy 3: Ordenamiento de productos según su demanda total**

Greedy (3)

Rehacer Greedy 1 y 2 ordenando los productos de mayor a menor según la demanda.

Partiendo de la solucion, veo si la puedo mejorar

En cada paso:
    Cambio el tipo de caja, y me fijo si con los descuentos puedo tener un menor costo

Restriccion: dim_actual <= dim_producto x 1.1

Headspace (grosor 4.5): dim_actual * 0.9 >= dim_producto

dim_actual >= dim_producto : 0.9
dim_actual <= dim_producto x 1.1

dim_producto x 1.11 <= dim_actual <= dim_producto x 1.1